<a href="https://colab.research.google.com/github/Nayab-khalid/FlyRank-AI-Internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nayab-khalid/FlyRank-AI-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [4]:
# ============================================================
# ML-04 — 1. UNIT OF ANALYSIS + TIME WINDOW
# ============================================================

%pip install -q duckdb huggingface_hub pandas

import duckdb
from google.colab import userdata

# Get Hugging Face token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

# Connect DuckDB
con = duckdb.connect()

# Authenticate with Hugging Face
con.execute(
    "CREATE SECRET hf_secret "
    "(TYPE HUGGINGFACE, TOKEN ?)",
    [HF_TOKEN]
)

# FlyRank warehouse
rel = "hf://datasets/FlyRank/internship-warehouse"

# ------------------------------------------------------------
# CONTRACT ANSWER
# ------------------------------------------------------------

print("""
UNIT OF ANALYSIS + TIME WINDOW

One row = one content item for one client on one report date
in the fact_content_daily_performance table.

I use March 2026 as the development window. March is a middle
month in the warehouse history, so I use it to develop and
verify the data contract rather than using the final month
as development data.
""")

# ------------------------------------------------------------
# VERIFY THE GRAIN AND TIME WINDOW
# ------------------------------------------------------------

query = f"""
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT
        CAST(report_date AS VARCHAR)
        || '|' || client_hash_id
        || '|' || content_hash_id
    ) AS distinct_grain_rows,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet',
    hive_partitioning = true
)
WHERE month = '2026-03'
"""

result = con.execute(query).df()

display(result)

print("\nClaim being verified:")
print("One row = report_date × client_hash_id × content_hash_id")

print("\nDevelopment window:")
print("March 2026")


UNIT OF ANALYSIS + TIME WINDOW

One row = one content item for one client on one report date
in the fact_content_daily_performance table.

I use March 2026 as the development window. March is a middle
month in the warehouse history, so I use it to develop and
verify the data contract rather than using the final month
as development data.



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,distinct_grain_rows,min_date,max_date
0,9841378,9841378,2026-03-01,2026-03-31



Claim being verified:
One row = report_date × client_hash_id × content_hash_id

Development window:
March 2026


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [5]:
# ============================================================
# ML-04 — 2. FIELDS: FEATURE / LABEL / CONTEXT / EXCLUDED
# ============================================================

%pip install -q duckdb huggingface_hub pandas

import duckdb
from google.colab import userdata

# Get Hugging Face token
HF_TOKEN = userdata.get("HF_TOKEN")

# Connect DuckDB
con = duckdb.connect()

# Authenticate
con.execute(
    "CREATE SECRET hf_secret "
    "(TYPE HUGGINGFACE, TOKEN ?)",
    [HF_TOKEN]
)

rel = "hf://datasets/FlyRank/internship-warehouse"

# ------------------------------------------------------------
# READ ACTUAL WAREHOUSE SCHEMA
# ------------------------------------------------------------

schema_query = f"""
DESCRIBE
SELECT *
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet',
    hive_partitioning = true
)
"""

schema = con.execute(schema_query).df()

print("=" * 70)
print("ACTUAL WAREHOUSE COLUMNS")
print("=" * 70)

display(schema[["column_name", "column_type"]])

# ------------------------------------------------------------
# CONTRACT
# ------------------------------------------------------------

print("""
FIELD CONTRACT

FEATURES:
I will use observable search/content performance signals that
are available before the decision point.

LABEL / PROXY:
The decline proxy is based on the documented trend outcome.
The starter pipeline defines decline as trend_direction = 'down'.

CONTEXT:
report_date, client_hash_id, and content_hash_id identify and
group observations. They are not predictive features.

EXCLUDED:
trend_direction and trend_pct are excluded from the feature set
because they define or encode the decline outcome and can cause
target leakage.

Client and content identifiers are also excluded as predictive
features because they are pseudonymous identifiers intended for
grouping and joins.
""")

# ------------------------------------------------------------
# SHOW LIKELY RELEVANT COLUMNS
# ------------------------------------------------------------

keywords = [
    "report",
    "date",
    "client",
    "content",
    "impression",
    "click",
    "session",
    "position",
    "trend",
    "ga4",
    "gsc",
    "update",
    "age"
]

relevant = schema[
    schema["column_name"]
    .str.lower()
    .apply(lambda x: any(k in x for k in keywords))
]

print("=" * 70)
print("RELEVANT WAREHOUSE COLUMNS TO REVIEW")
print("=" * 70)

display(relevant[["column_name", "column_type"]])


ACTUAL WAREHOUSE COLUMNS


,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT



FIELD CONTRACT

FEATURES:
I will use observable search/content performance signals that
are available before the decision point.

LABEL / PROXY:
The decline proxy is based on the documented trend outcome.
The starter pipeline defines decline as trend_direction = 'down'.

CONTEXT:
report_date, client_hash_id, and content_hash_id identify and
group observations. They are not predictive features.

EXCLUDED:
trend_direction and trend_pct are excluded from the feature set
because they define or encode the decline outcome and can cause
target leakage.

Client and content identifiers are also excluded as predictive
features because they are pseudonymous identifiers intended for
grouping and joins.

RELEVANT WAREHOUSE COLUMNS TO REVIEW


,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [6]:
# ============================================================
# ML-04 — 3. VERIFY IT WITH THREE QUERIES
# ============================================================

%pip install -q duckdb huggingface_hub pandas

import duckdb
from google.colab import userdata

# Get Hugging Face token
HF_TOKEN = userdata.get("HF_TOKEN")

# Connect DuckDB
con = duckdb.connect()

# Authenticate
con.execute(
    "CREATE SECRET hf_secret "
    "(TYPE HUGGINGFACE, TOKEN ?)",
    [HF_TOKEN]
)

rel = "hf://datasets/FlyRank/internship-warehouse"

source = f"""
read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet',
    hive_partitioning = true
)
"""

# ============================================================
# QUERY 1 — GRAIN
# ============================================================

print("=" * 70)
print("QUERY 1 — GRAIN CHECK")
print("=" * 70)

grain_query = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS duplicate_count
FROM {source}
WHERE month = '2026-03'
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
ORDER BY duplicate_count DESC
"""

grain_result = con.execute(grain_query).df()

display(grain_result)

print(
    f"Duplicate report_date × client × content combinations: "
    f"{len(grain_result)}"
)

# ============================================================
# QUERY 2 — ROW COUNT + DATE WINDOW
# ============================================================

print("=" * 70)
print("QUERY 2 — ROW COUNT AND DATE WINDOW")
print("=" * 70)

window_query = f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM {source}
WHERE month = '2026-03'
"""

window_result = con.execute(window_query).df()

display(window_result)

# ============================================================
# QUERY 3 — AVAILABILITY USING IS TRUE
# ============================================================

print("=" * 70)
print("QUERY 3 — AVAILABILITY USING IS TRUE")
print("=" * 70)

availability_query = f"""
SELECT
    COUNT(*) AS ga4_available_rows
FROM {source}
WHERE month = '2026-03'
  AND ga4_data_available IS TRUE
"""

availability_result = con.execute(
    availability_query
).df()

display(availability_result)

print("\nAvailability was checked with IS TRUE.")


QUERY 1 — GRAIN CHECK


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,duplicate_count


Duplicate report_date × client × content combinations: 0
QUERY 2 — ROW COUNT AND DATE WINDOW


,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


QUERY 3 — AVAILABILITY USING IS TRUE


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,ga4_available_rows
0,413966



Availability was checked with IS TRUE.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [7]:
# ============================================================
# ML-04 — 4. DATA LIMITS
# ============================================================

%pip install -q duckdb huggingface_hub pandas

import duckdb
from google.colab import userdata

# Get Hugging Face token
HF_TOKEN = userdata.get("HF_TOKEN")

# Connect DuckDB
con = duckdb.connect()

# Authenticate
con.execute(
    "CREATE SECRET hf_secret "
    "(TYPE HUGGINGFACE, TOKEN ?)",
    [HF_TOKEN]
)

rel = "hf://datasets/FlyRank/internship-warehouse"

source = f"""
read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet',
    hive_partitioning = true
)
"""

print("""
DATA LIMITS

The warehouse is an unbalanced panel, meaning different clients
have different amounts of historical coverage.

Some rows may have GSC data without GA4 data. Therefore, missing
GA4 measurements should not automatically be interpreted as
zero engagement.

The data is observational and supports measured, observed, and
directional decision-support findings. It cannot by itself prove
that refreshing a page caused its performance to improve.
""")

# Verify the three possible GA4 availability states
limits_query = f"""
SELECT
    COUNT(*) AS total_rows,

    COUNT(*) FILTER (
        WHERE ga4_data_available IS TRUE
    ) AS ga4_available_rows,

    COUNT(*) FILTER (
        WHERE ga4_data_available IS FALSE
    ) AS ga4_unavailable_rows,

    COUNT(*) FILTER (
        WHERE ga4_data_available IS NULL
    ) AS ga4_unknown_rows

FROM {source}
WHERE month = '2026-03'
"""

limits_result = con.execute(limits_query).df()

print("=" * 70)
print("GA4 AVAILABILITY STATES — MARCH 2026")
print("=" * 70)

display(limits_result)



DATA LIMITS

The warehouse is an unbalanced panel, meaning different clients
have different amounts of historical coverage.

Some rows may have GSC data without GA4 data. Therefore, missing
GA4 measurements should not automatically be interpreted as
zero engagement.

The data is observational and supports measured, observed, and
directional decision-support findings. It cannot by itself prove
that refreshing a page caused its performance to improve.



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

GA4 AVAILABILITY STATES — MARCH 2026


,total_rows,ga4_available_rows,ga4_unavailable_rows,ga4_unknown_rows
0,9841378,413966,6408671,3018741


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.